# Analisis estadistico avanzado para optimizar el uso de jugadores

Una vez ya obtenidos los jugadores con sus estadisticas es hora de ponernos a reflexionar que metricas podemos intentar sacar para obtener las mejores puntuaciones en base a las reglas del juego.

Las reglas de puntuacion son las siguientes:

- Punto anotado: Cada punto anotado valdra +1
- Asistencia: Cada asistencia registrada valdara +1.5
- Rebotes: Cada rebote resgistrado valdra +1.2
- Robo: Cada robo generado valdra +2
- Tapones: Cada tapon valdra +2
- Fallo de 2: Cada fallo de lanzamiento de 2 puntos valdran -0.3
- Fallo de 3: Cada fallo de lanzamiento de 3 puntos valdra -0.5
- Fallo de 1: Cada fallo de tiro libre valdra 0.5
- Perdida de balon: Cada perdida de balon valdra -2

Pero ademas hay que tener ciertas cuestiones como:

- Titular vs Suplente : No juega lo mismo un jugador que es titular que uno que es suplente, el nº de minutos se vera afectado
- Rol del jugador: Hay que entender que aporta en el sistema de juego del equipo, hay perfiles de rotacion, hay otros encargados de la anotacion, distribucion del juego, defensivos, otros hibridos como los 3&D, pivots con movilidad
- Record del equipo: hay jugadores que pueden generar mucho pero por el record del equipo, necesitan estar mas balanceados en la rotacion y otros donde el equipo puede no estar compitiendo, y por ello el jugador ser un gran generador.
- Conferencia y Division del equipo: Los equipos se dividen en 2 conferencias y estas a su vez en 4 divisiones. Esto afecta al nº de partidos que juegan contra x equipos y por tanto, facilidad de encontrarse a equipos con peores records que puedan permitir a los jugadores mejorar sus lineas estadisticas o viceversa.
- Opcion a premios individuales o All Star: Hay jugadores que generan mucho por intentar conseguir lograr galardones o participaciones que les reporten mejores premios economicos.
- ....

Por tanto vamos a generar una sistematizacion para ponderar estos criterios y entender que jugadores pueden ser interesantes.

Para poder realizar un calculo ponderado adecuado primero vamos a intentar resolver o encontrar informacion que nos pueda ayudar a indicar una ponderacion adecuada, por ver como puede afectar esto a la eleccion de un jugador.

### Ponderaciones: Como afecta ser titular a las estadisticas

Lo primero que vamos a comprobar son como puede afectar los minutos jugados por partido al jugador y como podria afectar esto a las estadisticas.

Podemos definir lo siguiente:

- Titular / Estrella (> 28 min/partido): El núcleo duro del equipo.
- Rotación (15 - 28 min/partido): El 6º hombre y los suplentes de calidad.
- Fondo de Armario (< 15 min/partido): Jugadores de rol específico o minutos de la basura.

Vamos a revisar cuantos jugadores hay de da tipo en primer lugar

In [4]:
import pandas as pd
import numpy as np
from pathlib import Path

# 0. Definimos la ruta donde guardaste el archivo en el Notebook 2
ruta_pickle = Path(r"C:\proyecto-faust\proyecto-nba-analytics\data\nba_processed_dataset.pkl")

# 1. Calculamos Minutos Por Partido (MPG)
# Evitamos división por cero si GP es 0
df['MPG'] = np.where(df['partidos_jugados'] > 0, 
                     df['minutos'] / df['partidos_jugados'], 
                     0)

# 2. Creamos la Clasificación de Roles (Etiquetado)
# Usamos np.select, que es la forma más rápida y limpia en Pandas (mejor que bucles if)
condiciones = [
    (df['MPG'] >= 28),                                      # Caso 1: Titular
    (df['MPG'] >= 15) & (df['MPG'] < 28),                   # Caso 2: Rotación
    (df['MPG'] < 15)                                        # Caso 3: Banquillo
]

etiquetas_roles = ['Titular', 'Rotacion', 'Banquillo']

df['ROL_INFERIDO'] = np.select(condiciones, etiquetas_roles, default='Sin_Minutos')

# 3. Validamos la distribución
print("\n--- DISTRIBUCIÓN DE JUGADORES POR ROL ESTIMADO ---")
print(df['ROL_INFERIDO'].value_counts())


--- DISTRIBUCIÓN DE JUGADORES POR ROL ESTIMADO ---
ROL_INFERIDO
Banquillo    208
Rotacion     200
Titular      122
Name: count, dtype: int64


Ahora incluyo esta informacion en el archivo sobre el que estoy trabajando

In [5]:
# 1. Validación Visual
print("--- MUESTRA DE VALIDACIÓN DE ROLES ---")
display(df[['nombre', 'minutos', 'MPG', 'ROL_INFERIDO']].sample(5))

# 2. Persistencia (Sobrescribiendo siempre el Mismo Archivo de Análisis)
# Usamos siempre este nombre para el archivo final con todas las métricas
nombre_archivo_final = "analytical_table.pkl"
ruta_destino = Path(r"C:\proyecto-faust\proyecto-nba-analytics\data") / nombre_archivo_final

df.to_pickle(ruta_destino)

print(f"\nArchivo actualizado: {nombre_archivo_final}")
print("Todas las nuevas métricas se han guardado en este único archivo.")

--- MUESTRA DE VALIDACIÓN DE ROLES ---


,nombre,minutos,MPG,ROL_INFERIDO
450,Garrett Temple,29,3.222222,Banquillo
458,Xavier Tillman,105,9.545455,Banquillo
300,Mac McClung,34,11.333333,Banquillo
260,Christian Koloko,152,15.200000,Rotacion
153,Anthony Gill,47,3.357143,Banquillo



Archivo actualizado: analytical_table.pkl
Todas las nuevas métricas se han guardado en este único archivo.
